# The Autonomous Database Architect
**The Future of RAG: From Read-Only to Read-Write Intelligence**

**Copyright 2026, Denis Rothman**

## **Overview**
In the previous 11 chapters, we built systems that **retrieved** existing data to answer questions. In this final chapter, we build an Autonomous Agent that **creates** new data structures on the fly.

We address a fundamental limitation of standard RAG: **"What if the data exists in raw text but not in a queryable format?"**

Instead of waiting for a human DBA to design a table, this "Autonomous Architect" analyzes unstructured text, designs a compliant Oracle 23ai schema (DDL), creates the table, and populates it with clean data (DML). This moves us from **Probabilistic RAG** (guessing answers from text) to **Deterministic RAG** (querying structured facts).

## **Notebook Roadmap**
This notebook implements the "Self-Evolving Database" pattern in four steps:

* **Step 1: Initialization**
    * Connect to the Oracle 23ai instance using the `OracleManager`.
    * Initialize the OpenAI client for the agent's cognitive functions.

* **Step 2: The Autonomous Architect (The Designer)**
    * Define an agent function that accepts a high-level goal (e.g., "Track project risks") and a raw text sample.
    * The agent autonomously generates and executes the SQL `CREATE TABLE` statement to build the necessary infrastructure.

* **Step 3: The ETL Worker (The Builder)**
    * Define an agent function that reads unstructured text chunks.
    * The agent transforms the text into structured SQL `INSERT` statements and populates the new table.

* **Step 4: The Autonomous Scenario**
    * We present the agent with a "Messy Log" scenario.
    * We watch as the agent autonomously builds its own memory structure and then queries it to provide a 100% accurate answer.

# 1.Installation and Setup

## Installing OpenAI

In [13]:
!pip install openai==2.15.0

In [14]:
import os
from openai import OpenAI
from google.colab import userdata

try:
    # Explicitly check for the mandatory OpenAI key
    openai_key = userdata.get("API_KEY")
    if not openai_key:
        raise ValueError("OpenAI API_KEY not found in Secrets.")

    os.environ["OPENAI_API_KEY"] = openai_key
    client = OpenAI()
    print("✅ OpenAI initialized (Mandatory).")

except Exception as e:
    print(f"❌ Critical Error: OpenAI initialization failed. {e}")
    # We raise the error here because OpenAI is required for the Planner to work at all
    raise e

✅ OpenAI initialized (Mandatory).


In [15]:
#@title Retrieving GitHub Private token (this cell will be deleted when the repository is public)
import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    pt = userdata.get("GITHUB_TOKEN")
    pt=pt.strip()
    if not pt:
        raise userdata.SecretNotFoundError("GITHUB_TOKEN not found.")

    print("GITHUB_TOKENAPI private token loaded successfully.")

except userdata.SecretNotFoundError:
    print('Secret "GITHUB_TOKEN" not found.')
    print('Please activate or add your GITHUB_TOKEN private token to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the GITHUB_TOKEN: {e}")

GITHUB_TOKENAPI private token loaded successfully.


## DBA Parameters



## 1.1 Global Configuration Flags

This section acts as the **Control Panel** for the notebook. These boolean flags determine which administrative tasks will execute during this run. This allows the DBA to run the notebook safely in "maintenance mode" without accidentally recreating tables or wiping data.

* **`unzip_wallet`**: Set to `True` only for the initial setup to extract the Oracle Wallet configuration. Once extracted, set to `False` to avoid redundant file operations.
* **`create_tables`**: Set to `True` to execute DDL statements. This initializes the `CONTEXT_LIBRARY` and `KNOWLEDGE_STORE` tables. (Keep `False` if tables already exist).
* **`empty_tables`**: Set to `True` to perform a **TRUNCATE** operation. **Warning:** This will permanently delete all vector data from the tables while preserving the structure.

In [16]:
unzip_wallet=False  # True to unzip the wallet. False to only unzip the wallet once

## 1.2. Oracle environment Setup & Wallet Extraction

This step prepares the runtime environment by connecting to Google Drive and ensuring the Oracle Wallet is available. The Oracle Wallet contains the SSL/TLS certificates and configuration files (tnsnames.ora, sqlnet.ora) necessary for a secure connection to the Oracle Autonomous Database.

Google Drive Mount: Maps your personal Drive to /content/drive, allowing the notebook to read credentials and configuration files stored externally.

Wallet Unzipping: If unzip_wallet is set to True, the script searches for the wallet ZIP file in the specified path and extracts it. This ensures the connection drivers can locate the required security certificates.

Path Definition: Sets wallet_path to the directory where the unzipped configuration files reside, which will be passed to the Oracle driver in subsequent steps.

In [17]:
from google.colab import drive
drive.mount('/content/drive')
if unzip_wallet==True:
  !unzip -o "/content/drive/MyDrive/files/oracle/Wallet_*.zip" -d "/content/drive/MyDrive/files/oracle"
wallet_path = "/content/drive/MyDrive/oracle_wallet"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.3. Install Oracle Database Driver
This command installs the oracledb Python driver, which is the renamed and modernized successor to the legacy cx_Oracle driver. It serves as the bridge between this Python notebook and the Oracle Database.

Version Pinning (==3.4.1): The version is explicitly fixed to 3.4.1 to ensure stability and reproducibility. In production DBA scripts, pinning versions prevents unexpected updates or API changes from breaking the automation workflow.

Thin Client Mode: By default, this driver operates in "Thin" mode, meaning it communicates directly with the database using pure Python code without requiring local Oracle Client libraries (Instant Client).

In [18]:
!pip install oracledb==3.4.1

In [19]:
import oracledb
print(oracledb.__version__)

3.4.1


## 1.4. Additional imports

In [20]:
# Imports for this notebook
import json
import time
from tqdm.auto import tqdm
import tiktoken                                                         # tokenization
from tenacity import retry, stop_after_attempt, wait_random_exponential # to retry functions
import re
import textwrap
from IPython.display import display, Markdown
import copy

## 1.5.Establish Secure Database Connection
This step handles the authentication and connection to the Oracle 23ai instance. It is designed to adhere to security best practices by strictly separating code from credentials.

External Credential Management: Instead of hardcoding sensitive passwords directly into the notebook (which is a security risk), the script reads a credentials.txt file stored securely on Google Drive.

Credential Parsing: The read_key_value_file helper function parses the external file to retrieve the username, password, Wallet password, and DSN (Data Source Name).

Connection Initialization: The oracledb.connect() method establishes the session using the extracted credentials and the local Wallet configuration path.

Connectivity Test: A simple "Heartbeat" query (SELECT ... FROM DUAL) is executed immediately to verify that the connection is active and stable before proceeding.

In [21]:
import os

creds_path = "/content/drive/MyDrive/files/oracle/credentials.txt"
if not os.path.exists(creds_path):
    raise FileNotFoundError(f"Credentials file not found: {creds_path}")

def read_key_value_file(path):
    creds = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            creds[k.strip()] = v.strip()
    return creds

creds = read_key_value_file(creds_path)

# Use values (do not print them)
user = creds.get("user")
password = creds.get("password")
wallet_password = creds.get("wallet_password")
dsn = creds.get("dsn", "mqyv1gnzq40yxs41_high")
wallet_path = creds.get("wallet_path", "/content/drive/MyDrive/files/oracle")

import oracledb
connection = oracledb.connect(
    user=user,
    password=password,
    dsn=dsn,
    config_dir=wallet_path,
    wallet_location=wallet_path,
    wallet_password=wallet_password
)

cursor = connection.cursor()
cursor.execute("SELECT 'Connected!' FROM dual")
print(cursor.fetchone())

('Connected!',)


# 2. The Autonomous Architect (The Designer)
In this step, we define the `autonomous_data_architect` function. This is the "Brain" of the operation.

**How it works:**
1.  **Perceive:** It accepts a high-level `goal` and a `raw_text_sample`.
2.  **Plan:** It prompts `gpt-5.2` to design an optimal Oracle 23ai table schema (DDL).
3.  **Act:** It executes the `CREATE TABLE` statement immediately.
4.  **Audit:** It logs the creation event in the `AGENT_SCHEMA_REGISTRY` for governance.

In [22]:
import json

# 1. Local Helper for JSON Generation (Replaces 'helpers.py')
def call_genai_json(system_prompt, user_prompt, client, model="gpt-5.2"):
    """
    A specialized version of your GenAI call that enforces JSON output.
    Essential for generating machine-readable SQL structures.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"}, # Force JSON mode
            temperature=0.1
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ GenAI Call Failed: {str(e)}")
        raise e

# 2. The Autonomous Architect Function
def autonomous_data_architect(goal, raw_text_sample, client):
    """
    Analyzes the User's Goal and Raw Data, then autonomously designs and builds
    an Oracle 23ai table using the existing database connection.
    """
    print(f"🏗️ [Architect] Analyzing goal: '{goal}'...")

    # --- Phase 1: Design (Generative) ---
    system_prompt = """
    You are an Oracle 23ai Database Architect.
    Task: Create a SQL table schema (DDL) to structure the provided raw text based on the user's goal.

    Output Format: JSON only.
    {
        "table_name": "AI_<meaningful_name>",
        "ddl_statement": "CREATE TABLE AI_... (...)"
    }

    Rules:
    1. Table name MUST start with 'AI_'.
    2. Use standard Oracle data types (VARCHAR2(4000), NUMBER, DATE, CLOB).
    3. Ensure the DDL is valid and runnable.
    """

    user_prompt = f"""
    GOAL: {goal}
    SAMPLE RAW DATA:
    {raw_text_sample[:1000]}

    Generate the JSON DDL now.
    """

    try:
        # Call GPT-5.2
        response_json = call_genai_json(system_prompt, user_prompt, client)
        design = json.loads(response_json)

        table_name = design['table_name']
        ddl_script = design['ddl_statement']

        print(f"   📝 Proposed Table: {table_name}")
        print(f"   📜 Generated DDL: {ddl_script}")

    except Exception as e:
        print(f"❌ [Architect] Design Phase Failed: {e}")
        return None

    # --- Phase 2: Build (Database Action) ---
    try:
        # Use the global 'connection' object directly
        with connection.cursor() as cursor:

            # A. Cleanup (Drop if exists for this demo)
            try:
                cursor.execute(f"DROP TABLE {table_name}")
                print(f"   Existing table {table_name} dropped.")
            except:
                pass

            # B. Execute the AI's DDL
            cursor.execute(ddl_script)
            print(f"   ✅ Table '{table_name}' created successfully in Oracle 23ai.")

            # C. Register with Governance Log
            try:
                cursor.execute("""
                    INSERT INTO agent_schema_registry (user_goal, target_table_name, ddl_script)
                    VALUES (:goal, :tbl, :ddl)
                """, [goal, table_name, ddl_script])
                print(f"   ✅ Action logged to AGENT_SCHEMA_REGISTRY.")

            except Exception as registry_error:
                # Self-Healing: If registry is missing, create it
                if "ORA-00942" in str(registry_error):
                    print("   ⚠️ Registry missing. Creating 'AGENT_SCHEMA_REGISTRY'...")
                    cursor.execute("""
                        CREATE TABLE agent_schema_registry (
                            registry_id NUMBER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
                            user_goal CLOB,
                            target_table_name VARCHAR2(128),
                            ddl_script CLOB,
                            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                        )
                    """)
                    # Retry Insert
                    cursor.execute("""
                        INSERT INTO agent_schema_registry (user_goal, target_table_name, ddl_script)
                        VALUES (:goal, :tbl, :ddl)
                    """, [goal, table_name, ddl_script])
                    print(f"   ✅ Registry created and action logged.")
                else:
                    print(f"   ⚠️ Warning: Could not log to registry: {registry_error}")

            # Commit changes
            connection.commit()
            return table_name

    except Exception as e:
        print(f"❌ [Architect] Construction Phase Failed: {e}")
        return None

# 3. The ETL Worker (The Builder)
In this step, we define the `autonomous_etl_worker`. While the Architect (Step 2) creates the empty container (the table), the ETL Worker fills it with value.

**How it works:**
1.  **Iterate:** It loops through the list of unstructured text chunks (`raw_data_list`).
2.  **Transform (Generative):** For each chunk, it calls the LLM (using our JSON helper) to generate a valid Oracle `INSERT` statement that maps the unstructured text to the new table's columns.
3.  **Load (DML Execution):** It executes the SQL immediately using the open database connection.
4.  **Commit:** It commits the transaction, permanently saving the structured knowledge.

In [25]:
def autonomous_etl_worker(table_name, raw_data_list, client):
    """
    The 'Builder' Agent.
    Reads unstructured text -> Inspects Table Schema -> Generates SQL INSERT -> Populates Table.
    """
    print(f"🔄 [ETL Worker] Starting construction on table '{table_name}'...")

    try:
        # Use the global 'connection' object directly
        with connection.cursor() as cursor:

            # --- SUB-STEP 1: Schema Reflection (The "Eyes") ---
            # The agent queries the DB to find out what columns actually exist.
            # This prevents hallucinations like using "SEVERITY" instead of "SEVERITY_CATEGORY".
            cursor.execute(f"""
                SELECT column_name, data_type
                FROM user_tab_columns
                WHERE table_name = :t
            """, [table_name.upper()])

            columns = cursor.fetchall()
            schema_desc = ", ".join([f"{c[0]} ({c[1]})" for c in columns])
            print(f"   👀 Detected Schema: {schema_desc}")

            if not columns:
                print(f"❌ Error: Table '{table_name}' not found or has no columns.")
                return

            # --- SUB-STEP 2: The Prompt ---
            system_prompt = f"""
            You are an Oracle 23ai Data Engineer.
            Task: Convert the provided Unstructured Data into a SQL INSERT statement for the table '{table_name}'.

            CRITICAL: You must ONLY use the columns listed in the target schema below. Do not invent columns.
            TARGET SCHEMA: {table_name} ({schema_desc})

            Output: JSON only {{ "sql": "INSERT INTO {table_name} ..." }}
            Rule: Ensure values are properly escaped.
            """

            # --- SUB-STEP 3: Loop and Load ---
            for i, text_chunk in enumerate(raw_data_list):
                user_prompt = f"DATA SEGMENT #{i+1}: {text_chunk}"

                # A. Generate SQL
                try:
                    response_json = call_genai_json(system_prompt, user_prompt, client)
                    sql_payload = json.loads(response_json)
                    insert_sql = sql_payload['sql']
                except Exception as llm_error:
                    print(f"   ⚠️ [ETL] Failed to generate SQL for segment #{i+1}: {llm_error}")
                    continue

                # B. Execute Insert
                try:
                    cursor.execute(insert_sql)
                    print(f"   -> Segment #{i+1} converted & inserted.")
                except Exception as db_error:
                    print(f"   ⚠️ [ETL] Database Insert Failed: {db_error}")
                    print(f"      Bad SQL: {insert_sql}")

            # 4. Commit the Build
            connection.commit()
            print(f"✅ [ETL Worker] Job Complete. Table '{table_name}' is ready for querying.")

    except Exception as e:
        print(f"❌ [ETL Worker] Process Failed: {e}")

# 4. The Autonomous Scenario: From Chaos to Order
In this final step, we simulate a common enterprise challenge: **"Data Blindness."**

We have a list of raw server logs (`raw_server_logs`). They contain critical information about downtime and severity, but because they are unstructured text, we cannot run SQL queries like *"Select all critical incidents"* or *"Sum the downtime minutes."*

**The Simulation:**
1.  **Goal:** We give the agent the goal: *"Analyze the downtime duration and severity."*
2.  **Architect:** The agent reads the logs, realizes it needs a table (e.g., `AI_INCIDENTS`) with specific columns (`SEVERITY`, `DOWNTIME_MINUTES`), and creates it.
3.  **Builder:** The agent extracts the data from the text and populates the table.
4.  **Verification:** Finally, we run a deterministic SQL query on the newly created table to prove that the unstructured data has been successfully transformed into structured knowledge.

In [26]:
# === The "Chapter 12" Autonomous Demo ===

# 1. The Chaos: Unstructured, messy server logs
raw_server_logs = [
    "Incident #101: Server Alpha experienced overheating. Resulted in downtime of 45 mins. Severity categorized as High.",
    "Incident #102: Network latency observed in region EU-West. No service interruption (0 mins downtime). Severity Low.",
    "Incident #103: Database deadlock detected on primary node. Service halted for 12 mins. Severity Critical.",
    "Incident #104: API Gateway timeout due to spike. Downtime 5 mins. Severity Medium."
]

goal = "Analyze downtime duration and severity for all incidents to identify critical risks."

print(f"🚀 [Simulation Start] User Goal: '{goal}'")
print("-" * 60)

# 2. The Architect: Design and Build the Memory
# The agent autonomously designs the table based on the logs and goal
target_table = autonomous_data_architect(goal, raw_server_logs[0], client)

if target_table:
    # 3. The Builder: Populate the Memory
    autonomous_etl_worker(target_table, raw_server_logs, client)

    # 4. The Result: Deterministic Accuracy
    print(f"\n🔮 [Verification] Querying the Agent's Created Table ({target_table})...")
    print("-" * 60)

    try:
        # Use the global connection object
        with connection.cursor() as cursor:
            # We use SELECT * to see exactly what the LLM designed.
            # We order by column 1 (usually ID) just to have a stable view.
            cursor.execute(f"SELECT * FROM {target_table} ORDER BY 1 DESC")

            rows = cursor.fetchall()

            # Fetch headers for display so we know what columns were invented
            if cursor.description:
                col_names = [row[0] for row in cursor.description]
                print(f"{' | '.join(col_names)}")
                print("-" * 60)

            for r in rows:
                print(r)

    except Exception as query_error:
        print(f"❌ Verification Query Failed: {query_error}")

print("\n✅ Simulation Complete. The Agent autonomously structured the unstructured.")

🚀 [Simulation Start] User Goal: 'Analyze downtime duration and severity for all incidents to identify critical risks.'
------------------------------------------------------------
🏗️ [Architect] Analyzing goal: 'Analyze downtime duration and severity for all incidents to identify critical risks.'...
   📝 Proposed Table: AI_INCIDENT_DOWNTIME
   📜 Generated DDL: CREATE TABLE AI_INCIDENT_DOWNTIME (
  INCIDENT_ID        NUMBER GENERATED BY DEFAULT AS IDENTITY,
  INCIDENT_NUMBER    NUMBER NOT NULL,
  INCIDENT_TEXT      CLOB,
  SYSTEM_NAME        VARCHAR2(4000),
  INCIDENT_DESCRIPTION CLOB,
  INCIDENT_DATE      DATE,
  DOWNTIME_MINUTES   NUMBER,
  SEVERITY           VARCHAR2(4000),
  CREATED_AT         DATE DEFAULT SYSDATE NOT NULL,
  CONSTRAINT PK_AI_INCIDENT_DOWNTIME PRIMARY KEY (INCIDENT_ID)
)
   Existing table AI_INCIDENT_DOWNTIME dropped.
   ✅ Table 'AI_INCIDENT_DOWNTIME' created successfully in Oracle 23ai.
   ✅ Action logged to AGENT_SCHEMA_REGISTRY.
🔄 [ETL Worker] Starting constructi

# 5. Governance: The DBA's View
With autonomous agents creating tables on the fly, governance is critical. We don't want a "Black Box."

In Step 2, our Architect Agent automatically logged its actions to the `AGENT_SCHEMA_REGISTRY`. Now, let's switch roles. We are no longer the Agent; we are the **DBA (Database Administrator)**.

We run a query to audit:
* **When** the table was created.
* **Why** it was created (the User Goal).
* **What** was created (the Table Name).

This demonstrates the core philosophy of Chapter 12: **Autonomous Innovation wrapped in Enterprise Control.**

In [27]:
# === MONITORING: The DBA Audit Log ===

print("👮‍♂️ [DBA Admin] Auditing Agent Activity...")
print("-" * 80)

try:
    with connection.cursor() as cursor:
        # Query the registry we created in Step 2
        cursor.execute("""
            SELECT created_at, target_table_name, user_goal
            FROM agent_schema_registry
            ORDER BY created_at DESC
            FETCH FIRST 5 ROWS ONLY
        """)

        rows = cursor.fetchall()

        print(f"{'TIMESTAMP':<28} | {'TABLE CREATED':<25} | {'USER GOAL (Why?)'}")
        print("-" * 80)

        for row in rows:
            timestamp, table_name, goal = row
            # Handle CLOB for the goal if necessary
            goal_text = goal.read() if hasattr(goal, 'read') else str(goal)

            # Truncate goal for display
            goal_display = (goal_text[:40] + '...') if len(goal_text) > 40 else goal_text

            print(f"{str(timestamp):<28} | {table_name:<25} | {goal_display}")

except Exception as e:
    print(f"❌ Audit Failed: {e}")

print("-" * 80)
print("✅ Chapter 12 Complete: The loop from Unstructured Text to Governance is closed.")

👮‍♂️ [DBA Admin] Auditing Agent Activity...
--------------------------------------------------------------------------------
TIMESTAMP                    | TABLE CREATED             | USER GOAL (Why?)
--------------------------------------------------------------------------------
2026-01-13 22:13:07.935037   | AI_INCIDENT_DOWNTIME      | Analyze downtime duration and severity f...
2026-01-13 22:09:57.978197   | AI_INCIDENT_DOWNTIME      | Analyze downtime duration and severity f...
--------------------------------------------------------------------------------
✅ Chapter 12 Complete: The loop from Unstructured Text to Governance is closed.


## **Conclusion**
We have reached the end of our journey.

In this notebook, we broke the "Read-Only" barrier of traditional RAG. We proved that an AI agent, powered by Oracle 23ai and Large Language Models, can:
1.  **Perceive** a data gap in the enterprise.
2.  **Architect** a structural solution (DDL) to fill that gap.
3.  **Execute** the construction and population of that structure (DML).
4.  **Operate** safely under the governance of a DBA registry.

This creates a new paradigm for the future of work: **The Self-Evolving Database.**

Your database is no longer just a static container for data; it is now an active participant in your business logic. As you deploy these systems, remember the lessons of this book: **Context is King, Vectors are the Bridge, and Governance is the Key.**



# 6. Environment Cleanup (Optional)
As a responsible DBA, once the analysis is complete and the insights are extracted, we should drop the temporary structure to free up resources.

**Note:** We will **drop the table** (`AI_INCIDENT_DOWNTIME`) but we will **keep the registry entry** (`AGENT_SCHEMA_REGISTRY`). This ensures we maintain a permanent audit trail of what the AI did, even after the data is gone.

In [28]:
# === 6. Cleanup & Reset ===

print(f"🧹 [DBA] Cleaning up temporary resources...")

try:
    with connection.cursor() as cursor:
        # 1. Drop the AI-created table
        # We use the 'target_table' variable from Step 4
        if 'target_table' in globals() and target_table:
            try:
                cursor.execute(f"DROP TABLE {target_table}")
                print(f"   🗑️ Dropped Table: {target_table}")
            except Exception as e:
                print(f"   ⚠️ Could not drop table: {e}")
        else:
            print("   ℹ️ No table target found to drop.")

        # 2. Commit the cleanup
        connection.commit()

    print("\n✅ Environment Reset. The Audit Log (AGENT_SCHEMA_REGISTRY) remains for compliance.")

except Exception as e:
    print(f"❌ Cleanup Failed: {e}")

🧹 [DBA] Cleaning up temporary resources...
   🗑️ Dropped Table: AI_INCIDENT_DOWNTIME

✅ Environment Reset. The Audit Log (AGENT_SCHEMA_REGISTRY) remains for compliance.


# 7. Final Proof: The Persistent Audit Trail
We claimed in Step 6 that the "Audit Log remains." Let's prove it.

We have just dropped the `AI_INCIDENT_DOWNTIME` table. The data is gone. However, if we query the `AGENT_SCHEMA_REGISTRY` now, we should still see the record of the Agent's activity.

This demonstrates the separation of **Ephemeral Memory** (the agent's working tables) and **Persistent Governance** (the DBA's logs). Even if the agent runs 100 times and cleans up after itself 100 times, the DBA will have 100 distinct log entries.

In [29]:
# === 7. Final Proof ===

print("📜 [Final Audit] Verifying Registry Persistence after Cleanup...")
print("-" * 80)

try:
    with connection.cursor() as cursor:
        # 1. Verify the Data Table is GONE
        try:
            cursor.execute("SELECT count(*) FROM AI_INCIDENT_DOWNTIME")
            print("❌ Warning: Table AI_INCIDENT_DOWNTIME still exists!")
        except Exception as e:
            if "ORA-00942" in str(e):
                print("✅ Proof: Table 'AI_INCIDENT_DOWNTIME' is truly deleted (ORA-00942).")
            else:
                print(f"   (Unexpected error checking table: {e})")

        print("-" * 80)

        # 2. Verify the Audit Log is STILL THERE
        cursor.execute("SELECT count(*) FROM agent_schema_registry")
        count = cursor.fetchone()[0]
        print(f"✅ Proof: The Registry still contains {count} historical records.")
        print("")

        # Show the latest entries
        cursor.execute("""
            SELECT created_at, target_table_name, user_goal
            FROM agent_schema_registry
            ORDER BY created_at DESC
            FETCH FIRST 3 ROWS ONLY
        """)

        rows = cursor.fetchall()
        for row in rows:
            timestamp, table_name, goal = row
            # Handle CLOB if needed
            goal_str = goal.read() if hasattr(goal, 'read') else str(goal)
            print(f"   🕒 {timestamp} | Created: {table_name} | Goal: {goal_str[:30]}...")

except Exception as e:
    print(f"❌ Verification Failed: {e}")

📜 [Final Audit] Verifying Registry Persistence after Cleanup...
--------------------------------------------------------------------------------
✅ Proof: Table 'AI_INCIDENT_DOWNTIME' is truly deleted (ORA-00942).
--------------------------------------------------------------------------------
✅ Proof: The Registry still contains 2 historical records.

   🕒 2026-01-13 22:13:07.935037 | Created: AI_INCIDENT_DOWNTIME | Goal: Analyze downtime duration and ...
   🕒 2026-01-13 22:09:57.978197 | Created: AI_INCIDENT_DOWNTIME | Goal: Analyze downtime duration and ...
